# Experiment 08 — The Same Conversation, on Biological Neurons

Experiment 07 wired up a conversational agent and trained it. It used
`nn.Linear`, `nn.Embedding`, `nn.GRUCell` — standard artificial neurons,
trained by backpropagation. When asked directly, the honest answer was: no,
that isn't experiment 01's biological neuron (leaky integrate-and-fire,
learning via Hebbian plasticity, no backward pass). This notebook is the
actual answer to "can you build it on that instead" — the same task, the
same 14-turn dataset, the same tests, but every classifier is a population of
experiment 01's exact `Neuron` class, trained with a supervised Hebbian rule
instead of Adam + cross-entropy.

**Set expectations going in:** Hebbian learning has no error-correction
signal and no established mechanism for autoregressive word-by-word
generation. Rather than fake that with something that isn't really Hebbian,
two real adaptations are made and flagged clearly:
1. Classification (intent, emotion, tone, plan) becomes **spike-count
   voting**: one neuron per class, the one that fires most during a fixed
   presentation window wins.
2. "Generation" becomes **associative recall**: a Hebbian memory that stores
   the 14 training responses and retrieves whichever one best matches the
   current situation. It cannot compose a sentence it wasn't trained on —
   only recognize which trained one fits. That's a real, different
   capability than experiment 07's GRU, and the comparison at the end is
   built to make that difference visible, not hide it.

In [1]:
import math
import random

random.seed(0)

## The neuron: identical to experiment 01

Same leaky integrate-and-fire `Neuron` class, unchanged — membrane
potential, exponential decay toward rest, threshold, a reset undershoot,
refractory period. The only new piece is `clamped_hebbian_update`: where
experiment 01's plain Hebbian rule only strengthened a synapse when the
neuron *actually* spiked, this version can be told "the correct answer is
this neuron" and strengthen it regardless of whether it fired on its own
yet — the standard way a Hebbian/associative memory is trained with a
teaching signal (the same principle behind classic correlation-matrix
memories and Hopfield network storage).

In [2]:
class Neuron:
    """Identical to experiment 01's leaky integrate-and-fire neuron."""

    def __init__(self, n_inputs, weights=None, threshold=1.0, rest=0.0, reset=-0.1,
                 tau_m=20.0, dt=1.0, refractory_ms=3.0):
        # Zero-initialized, not random -- this population of neurons IS an associative
        # memory, and correlation-matrix / Hopfield memories are conventionally built
        # up entirely from Hebbian updates rather than a random starting point.
        self.weights = list(weights) if weights is not None else [0.0] * n_inputs
        self.threshold = threshold
        self.rest = rest
        self.reset = reset
        self.tau_m = tau_m
        self.dt = dt
        self.refractory_steps = round(refractory_ms / dt)
        self.v = rest
        self.refractory_timer = 0
        self.last_input = [0.0] * n_inputs
        self.spiked = False
        self.decay = math.exp(-dt / tau_m)

    def step(self, inputs):
        self.last_input = list(inputs)
        if self.refractory_timer > 0:
            self.refractory_timer -= 1
            self.v = self.reset
            self.spiked = False
            return 0
        self.v = self.rest + (self.v - self.rest) * self.decay
        self.v += sum(w * x for w, x in zip(self.weights, inputs))
        if self.v >= self.threshold:
            self.v = self.reset
            self.refractory_timer = self.refractory_steps
            self.spiked = True
            return 1
        self.spiked = False
        return 0


def clamped_hebbian_update(neuron, lr, w_max=1.0):
    """Force post=1 (this IS the correct answer) regardless of whether the
    neuron actually spiked -- supervised/associative-memory Hebbian training."""
    for i in range(len(neuron.weights)):
        neuron.weights[i] += lr * neuron.last_input[i]
        neuron.weights[i] = min(neuron.weights[i], w_max)

## A Hebbian classifier: one neuron per class, spike-count voting

Present an input pattern for `T` timesteps to one neuron per class. During
training, the correct class's neuron gets a clamped Hebbian update at every
timestep the input is active; the other neurons are left untouched (a clean,
one-shot associative update — no competitive/anti-Hebbian term). During
classification, present the pattern again and count spikes; whichever neuron
fired most is the answer.

**A real tuning trap worth keeping, not hiding:** the first version used
`lr=0.1`, and every active-word weight hit `w_max=1.0` — saturated — within
a *single* training presentation (`T=10` steps × `lr=0.1` = 1.0). Once
saturated, unrelated classes that happen to share common words end up with
nearly identical spike counts, and accuracy collapses toward chance. Dropping
to `lr=0.002` for the fused-input classifiers leaves real headroom for
distinctions to survive — a direct, mechanistic illustration of why Hebbian
learning without any error-correcting or normalizing term is much more
sensitive to hyperparameters than gradient descent.

In [3]:
class HebbianClassifier:
    def __init__(self, n_inputs, class_names, T=10, lr=0.1, w_max=1.0):
        self.class_names = class_names
        self.T = T
        self.lr = lr
        self.w_max = w_max
        self.neurons = [Neuron(n_inputs=n_inputs) for _ in class_names]

    def _reset(self):
        for n in self.neurons:
            n.v = n.rest
            n.refractory_timer = 0

    def train_example(self, pattern, correct_class):
        self._reset()
        correct_idx = self.class_names.index(correct_class)
        for _ in range(self.T):
            for neuron in self.neurons:
                neuron.step(pattern)
            clamped_hebbian_update(self.neurons[correct_idx], lr=self.lr, w_max=self.w_max)

    def classify(self, pattern):
        self._reset()
        spike_counts = [0] * len(self.neurons)
        for _ in range(self.T):
            for idx, neuron in enumerate(self.neurons):
                spike_counts[idx] += neuron.step(pattern)
        predicted_idx = spike_counts.index(max(spike_counts))
        return self.class_names[predicted_idx], spike_counts

## The exact same 14-turn dataset as experiment 07

Byte-for-byte the same conversations, labels, and the same deliberate
"okay"-appears-twice-with-different-context test case — the whole point is
a fair, direct comparison, not a different, easier task.

In [4]:
train_conversations = [
    [   # A: friendly small talk
        dict(user="hello there friend", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="hello it is good to see you"),
        dict(user="how are you today", intent="question", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="i am doing well thank you"),
        dict(user="nice to meet you", intent="greeting", emotion="happy", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="nice to meet you too"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="playful", plan="answer_directly", response="great let us continue"),
    ],
    [   # B: formal question/answer
        dict(user="what time is the meeting", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="answer_directly", response="the meeting starts at three"),
        dict(user="where is the file", intent="question", emotion="neutral", formality=0.8, closeness=0.3, urgency=0.3,
             tone="formal", plan="ask_clarifying_question", response="which file do you mean"),
    ],
    [   # C: distress -> empathize (the "okay" contrast case lives here)
        dict(user="i am really stressed about this", intent="statement", emotion="anxious", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="that sounds really hard"),
        dict(user="i do not know what to do", intent="statement", emotion="sad", formality=0.3, closeness=0.7, urgency=0.5,
             tone="supportive", plan="empathize", response="i hear you and that matters"),
        dict(user="okay", intent="statement", emotion="neutral", formality=0.2, closeness=0.6, urgency=0.1,
             tone="supportive", plan="empathize", response="take your time i am here for you"),
        dict(user="thank you for listening", intent="statement", emotion="happy", formality=0.3, closeness=0.7, urgency=0.2,
             tone="playful", plan="answer_directly", response="i am glad i could help"),
    ],
    [   # D: commands
        dict(user="please close the door", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="closing the door now"),
        dict(user="turn off the lights", intent="command", emotion="neutral", formality=0.6, closeness=0.4, urgency=0.4,
             tone="formal", plan="give_instruction", response="turning off the lights"),
    ],
    [   # E: ambiguous -> clarify -> instruct
        dict(user="can you fix it", intent="question", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.5,
             tone="urgent", plan="ask_clarifying_question", response="which one do you mean"),
        dict(user="the printer upstairs", intent="statement", emotion="neutral", formality=0.4, closeness=0.5, urgency=0.6,
             tone="urgent", plan="give_instruction", response="restart the device now"),
    ],
]

probe_conversation = [
    dict(user="this is not working at all", formality=0.3, closeness=0.4, urgency=0.7),
    dict(user="still broken", formality=0.3, closeness=0.4, urgency=0.8),
]

intents = ["question", "statement", "greeting", "command"]
emotions = ["neutral", "happy", "sad", "angry", "anxious"]
tones = ["supportive", "formal", "playful", "urgent"]
plans = ["answer_directly", "ask_clarifying_question", "empathize", "give_instruction"]
responses = [t["response"] for conv in train_conversations for t in conv]  # 14 unique strings

real_words = sorted({w for conv in train_conversations for t in conv for w in t["user"].split()})
word_to_idx = {w: i for i, w in enumerate(real_words)}
n_turns = sum(len(c) for c in train_conversations)
print(f"{n_turns} turns, {len(real_words)} distinct user-utterance words, {len(responses)} unique responses")


def word_pattern(sentence):
    """Bag-of-words spike pattern: 1 for each known word present, order discarded --
    the population-coding analog of experiment 02's mean-pooling."""
    v = [0.0] * len(real_words)
    for w in sentence.split():
        if w in word_to_idx:
            v[word_to_idx[w]] = 1.0
    return v

14 turns, 41 distinct user-utterance words, 14 unique responses


## Architecture: classifiers in, associative memory out

- **Comprehension** = two `HebbianClassifier`s (intent, emotion) reading the
  raw word-presence pattern directly — no learned embedding layer, since
  there's no backward pass to train one against a downstream loss.
- **Memory** = a leaky trace: `memory = memory * decay + this_turn's_meaning`,
  the same exponential-leak idea as the membrane potential itself, just kept
  at the population level instead of inside one neuron.
- **Fusion** = concatenation only (`meaning ⧺ memory ⧺ emotion-one-hot ⧺
  social`) — no learned projection layer, for the same reason as above.
- **Planning** = two more `HebbianClassifier`s (tone, plan) reading the
  fused vector.
- **Response** = a `HebbianClassifier` with one neuron **per training
  response** (14 classes, one real example each) reading `fused ⧺
  plan-one-hot`. This is explicitly pattern *recall*, not word-by-word
  generation.

In [5]:
MEMORY_DECAY = 0.5
FUSED_LR = 0.002  # tuned down from 0.1 to avoid the saturation trap described above


class HebbianAgent:
    def __init__(self):
        self.intent_clf = HebbianClassifier(len(real_words), intents, T=10, lr=0.1, w_max=1.0)
        self.emotion_clf = HebbianClassifier(len(real_words), emotions, T=10, lr=0.1, w_max=1.0)
        fused_dim = len(real_words) * 2 + len(emotions) + 3
        self.tone_clf = HebbianClassifier(fused_dim, tones, T=10, lr=FUSED_LR, w_max=1.0)
        self.plan_clf = HebbianClassifier(fused_dim, plans, T=10, lr=FUSED_LR, w_max=1.0)
        self.response_clf = HebbianClassifier(fused_dim + len(plans), list(range(len(responses))),
                                               T=10, lr=FUSED_LR, w_max=1.0)
        self.memory_dim = len(real_words)

    def _fused_pattern(self, meaning, memory, emotion_idx, social):
        emo_onehot = [0.0] * len(emotions)
        emo_onehot[emotion_idx] = 1.0
        return meaning + memory + emo_onehot + list(social)

    def step(self, turn, memory, teacher_force):
        """Same contract as experiment 07's agent.step(): teacher_force=True trains
        on ground-truth labels; False is fully autonomous (own predictions only)."""
        meaning = word_pattern(turn["user"])
        social = [turn["formality"], turn["closeness"], turn["urgency"]]

        if teacher_force:
            self.intent_clf.train_example(meaning, turn["intent"])
            self.emotion_clf.train_example(meaning, turn["emotion"])
            emotion_idx = emotions.index(turn["emotion"])
        else:
            predicted_emotion, _ = self.emotion_clf.classify(meaning)
            emotion_idx = emotions.index(predicted_emotion)

        fused = self._fused_pattern(meaning, memory, emotion_idx, social)

        if teacher_force:
            self.tone_clf.train_example(fused, turn["tone"])
            self.plan_clf.train_example(fused, turn["plan"])
            plan_idx = plans.index(turn["plan"])
        else:
            predicted_plan, _ = self.plan_clf.classify(fused)
            plan_idx = plans.index(predicted_plan)

        plan_onehot = [0.0] * len(plans)
        plan_onehot[plan_idx] = 1.0
        response_query = fused + plan_onehot

        if teacher_force:
            self.response_clf.train_example(response_query, responses.index(turn["response"]))

        predicted_intent, _ = self.intent_clf.classify(meaning)
        predicted_emotion, _ = self.emotion_clf.classify(meaning)
        predicted_tone, _ = self.tone_clf.classify(fused)
        predicted_plan, _ = self.plan_clf.classify(fused)
        predicted_response_idx, _ = self.response_clf.classify(response_query)

        new_memory = [m * MEMORY_DECAY + x for m, x in zip(memory, meaning)]

        return new_memory, dict(
            predicted_intent=predicted_intent, predicted_emotion=predicted_emotion,
            predicted_tone=predicted_tone, predicted_plan=predicted_plan,
            predicted_response=responses[predicted_response_idx],
        )


agent = HebbianAgent()
n_params = sum(len(n.weights) for clf in
               [agent.intent_clf, agent.emotion_clf, agent.tone_clf, agent.plan_clf, agent.response_clf]
               for n in clf.neurons)
print(f"agent parameter count: {n_params:,} (experiment 07's backprop version: 15,837)")

agent parameter count: 2,405 (experiment 07's backprop version: 15,837)


## Training: one clean pass

Walk through every conversation once, `teacher_force=True`, memory carried
within a conversation and reset between them — mechanically identical to
experiment 07's training loop. There's no optimizer, no loss to sum, no
`.backward()`: each Hebbian update is applied immediately as its turn is
presented.

In [6]:
train_log = []
for conv in train_conversations:
    memory = [0.0] * agent.memory_dim
    for turn in conv:
        memory, result = agent.step(turn, memory, teacher_force=True)
        train_log.append((turn, result, list(memory)))

print(f"trained on {len(train_log)} turns")

trained on 14 turns


## Evaluating fairly

One subtlety worth being explicit about: reading off predictions *during*
the training loop above would flatter the model, since each prediction
would come from weights that had just been updated on that exact example a
moment earlier. A fair read needs a **separate pass**, after all training
is done, in fully autonomous mode (`teacher_force=False` — own predictions
throughout, no more learning).

In [7]:
eval_log = []
for conv in train_conversations:
    memory = [0.0] * agent.memory_dim
    for turn in conv:
        memory, result = agent.step(turn, memory, teacher_force=False)
        eval_log.append((turn, result))

correct = dict(intent=0, emotion=0, tone=0, plan=0, response=0)
for turn, result in eval_log:
    correct["intent"] += result["predicted_intent"] == turn["intent"]
    correct["emotion"] += result["predicted_emotion"] == turn["emotion"]
    correct["tone"] += result["predicted_tone"] == turn["tone"]
    correct["plan"] += result["predicted_plan"] == turn["plan"]
    correct["response"] += result["predicted_response"] == turn["response"]

print("Hebbian agent (this notebook)  vs  experiment 07's backprop agent:")
backprop_baseline = dict(intent=1.00, emotion=1.00, tone=1.00, plan=1.00, response=1.00)
for k, v in correct.items():
    print(f"  {k:9} {v}/{n_turns} = {v / n_turns:.0%}   (backprop: {backprop_baseline[k]:.0%})")

Hebbian agent (this notebook)  vs  experiment 07's backprop agent:
  intent    13/14 = 93%   (backprop: 100%)
  emotion   14/14 = 100%   (backprop: 100%)
  tone      11/14 = 79%   (backprop: 100%)
  plan      10/14 = 71%   (backprop: 100%)
  response  13/14 = 93%   (backprop: 100%)


In [8]:
print("per-turn breakdown:\n")
for turn, result in eval_log:
    flags = []
    for k in ["intent", "emotion", "tone", "plan"]:
        if result["predicted_" + k] != turn[k]:
            flags.append(f"{k}: pred={result['predicted_' + k]} true={turn[k]}")
    if result["predicted_response"] != turn["response"]:
        flags.append(f"response: pred={result['predicted_response']!r} true={turn['response']!r}")
    print(f"{turn['user']!r:35} {'ALL OK' if not flags else ' | '.join(flags)}")

per-turn breakdown:

'hello there friend'                ALL OK
'how are you today'                 ALL OK
'nice to meet you'                  intent: pred=statement true=greeting
'okay'                              ALL OK
'what time is the meeting'          ALL OK
'where is the file'                 plan: pred=answer_directly true=ask_clarifying_question
'i am really stressed about this'   ALL OK
'i do not know what to do'          ALL OK
'okay'                              plan: pred=answer_directly true=empathize
'thank you for listening'           tone: pred=supportive true=playful
'please close the door'             ALL OK
'turn off the lights'               ALL OK
'can you fix it'                    tone: pred=formal true=urgent | plan: pred=answer_directly true=ask_clarifying_question | response: pred='hello it is good to see you' true='which one do you mean'
'the printer upstairs'              tone: pred=formal true=urgent | plan: pred=answer_directly true=give_instruction


## Does memory actually change anything? (the same test as experiment 07)

The identical "okay" pair: same text, same predicted emotion, same social
features, only the preceding conversation differs. `memory_after_t2` is
pulled directly from the training pass above (the real memory state at that
point) — not recomputed by replaying the turns, which would silently apply
extra Hebbian updates and quietly corrupt the comparison. (That's a real
mistake this notebook's development actually made and caught — worth the
one-line warning.)

In [9]:
memory_after_t2 = train_log[7][2]  # conversation C, right after "i do not know what to do"
okay_turn = train_conversations[2][2]

_, result_carried = agent.step(okay_turn, memory_after_t2, teacher_force=False)
_, result_reset = agent.step(okay_turn, [0.0] * agent.memory_dim, teacher_force=False)

print(f"turn: {okay_turn['user']!r}  (true target response: {okay_turn['response']!r})\n")
print("memory carried (real distress context):")
print(f"  tone={result_carried['predicted_tone']:10} plan={result_carried['predicted_plan']:24} response={result_carried['predicted_response']!r}")
print("memory reset (as if the prior turns never happened):")
print(f"  tone={result_reset['predicted_tone']:10} plan={result_reset['predicted_plan']:24} response={result_reset['predicted_response']!r}")

turn: 'okay'  (true target response: 'take your time i am here for you')

memory carried (real distress context):
  tone=supportive plan=answer_directly          response='take your time i am here for you'
memory reset (as if the prior turns never happened):
  tone=supportive plan=answer_directly          response='hello it is good to see you'


## A conversation it never saw

The exact same held-out probe as experiment 07, fully autonomous.

In [10]:
memory = [0.0] * agent.memory_dim
for turn in probe_conversation:
    memory, result = agent.step(turn, memory, teacher_force=False)
    print(f"{turn['user']!r:30} -> emotion={result['predicted_emotion']:8} tone={result['predicted_tone']:10} "
          f"plan={result['predicted_plan']:24} response={result['predicted_response']!r}")

'this is not working at all'   -> emotion=neutral  tone=formal     plan=answer_directly          response='hello it is good to see you'
'still broken'                 -> emotion=neutral  tone=formal     plan=answer_directly          response='hello it is good to see you'


## What actually happened, compared honestly to experiment 07

| task | this notebook (Hebbian) | experiment 07 (backprop) |
|---|---|---|
| intent | 93% (13/14) | 100% |
| emotion | 100% (14/14) | 100% |
| tone | 79% (11/14) | 100% |
| plan | 71% (10/14) | 100% |
| response | 93% (13/14) | 100% |

**Classification degrades gracefully, generation does not — or rather, there
is no generation, only recall.** Emotion held perfectly (5-way, from bag-of-
words alone); intent, tone, and plan all land well above chance (25% for
4-way, 20% for 5-way) but meaningfully below backprop's 100% everywhere —
exactly the expected cost of a learning rule with no error-correcting
signal: each update only ever asks "was this the right answer," never "how
wrong was I and in which direction." The per-turn breakdown shows genuinely
interesting failures too — e.g. `plan: pred=answer_directly true=empathize`
on `okay` in the distress conversation, which the memory-ablation test right
below explains directly.

**Memory measurably matters, through a completely different mechanism.**
The identical "okay" pair: with the real distress-conversation memory
carried in, the response classifier correctly recalled `take your time i am
here for you` — the true trained target. With memory forcibly zeroed
(same text, same predicted emotion, same social features), it recalled
`hello it is good to see you` instead — a different stored memory
entirely. Interesting nuance: `tone` and `plan` gave the *same* answer
(`supportive`, `answer_directly`) in both conditions — it's specifically the
14-way response-recall associative memory, reading `memory` directly as
part of its input pattern, where the effect shows up. A leaky trace feeding
a linear associative memory produces the same qualitative result as a
trained recurrent cell: persistent state changes behavior, verified by an
isolated, controlled comparison, not just decoration.

**The untrained probe is where the two architectures diverge hardest.**
Both novel lines got the same prediction: `emotion=neutral`, `tone=formal`,
`plan=answer_directly`, and — tellingly — the *exact same* recalled
response both times: `hello it is good to see you`, the greeting reply,
for a conversation about something broken. That's not a near-miss, it's the
associative memory falling back to whatever stored pattern is the nearest
match in a high-dimensional bag-of-words space when nothing fits well —
and a greeting reply for "this is not working at all" makes that failure
mode visible immediately. Experiment 07's backprop version, tested on this
exact same probe, produced `ask_clarifying_question` -> "which one do you
mean" then `give_instruction` -> "restart the device now": plausible and
on-topic, not verbatim-correct either, but coherent in a way this notebook's
version isn't. That gap is the real finding here: pure Hebbian association
has no mechanism to interpolate toward something new — it can only recall
one of its 14 memorized answers, correctly or not, and has no way to
"almost" get it right. Getting genuine compositional generation out of
spiking/Hebbian networks is a real, open research problem — the field
mostly falls back on surrogate-gradient backprop for exactly this reason —
not a gap this notebook's scope could close.